In [ ]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

from rfml_uav.nn.models import get_model  # Assuming this returns a PyTorch model

# Load environment variables
load_dotenv()
data_path = os.path.join(os.getenv("DATA_PATH"), "balanced")

# Enabling GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
np.random.seed(1)
epochs = 50
batch_size = 32

# KFold cross-validation
k_folds = 5
kf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

## Functions

In [3]:
def decode(datum):
    y = np.zeros((datum.shape[0], 1))
    for i in range(datum.shape[0]):
        y[i] = np.argmax(datum[i])
    return y

def encode(datum, num_classes):
    return np.eye(num_classes, dtype="uint8")[datum]

## Model training

In [ ]:
def train_model(feature: str, model_name: str):
    print(f"Training the {model_name} that uses \"{feature}\" dataset.")
    print(f"----------------------------------------------------------")
    results_path = os.path.join(os.getenv("DATA_PATH"), "results", feature, model_name)
    os.makedirs(results_path, exist_ok=True)

    inputs = np.load(os.path.join(data_path, f"{feature}.npz"))
    X = inputs["X"]
    y = inputs["y"].astype("uint8")

    cvscores = []

    # Create the dataset and DataLoader
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))

    for fold, (train, test) in enumerate(kf.split(X, decode(y))):
        print(f"Training fold {fold + 1}/{k_folds}")
        
        # Split data into training and validation sets

        train_subsampler = torch.utils.data.Subset(dataset, train)
        test_subsampler = torch.utils.data.Subset(dataset, test)

        train_loader = DataLoader(train_subsampler, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_subsampler, batch_size=batch_size, shuffle=False)

        print(len(train), len(test))
        print(X.shape[1])

        # Initialize the model
        model = get_model(model_name, input_dim=X.shape[1], n_classes=len(set(y))).to(device)
        

        # Loss function and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        # Training loop
        for epoch in range(epochs): 
            model.train()
            for (inputs, targets) in tqdm(train_loader):
                inputs, targets = inputs.unsqueeze(1).to(device), targets.to(device) # Add a dimension for channels
                
                optimizer.zero_grad()
                output = model(inputs)  
                loss = criterion(output, targets)
                loss.backward()
                optimizer.step()

            print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.unsqueeze(1).to(device), targets.to(device) # Add a dimension for channels
                
                output = model(inputs) 
                _, predicted = torch.max(output, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()

        accuracy = 100 * correct / total

        print(f"Validation Accuracy for fold {fold + 1}: {accuracy:.2f}%")
        cvscores.append(accuracy)

        # Predict on test set
        model.eval()
        y_pred = []
        with torch.no_grad():
            for inputs, _ in test_loader:
                inputs = inputs.unsqueeze(1).to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                y_pred.extend(predicted.cpu().numpy())

        np.savetxt(
            os.path.join(results_path, f"Results_{fold}.csv"),
            np.column_stack((y[test], y_pred)),
            delimiter=",",
            fmt="%s",
        )
        print(f"----------------------------------------------------------")

In [5]:
train_model("Wavelets", "RNN")

Training the RNN that uses "Wavelets" dataset.
----------------------------------------------------------
Training fold 1/5
359098 89775
256


11222it [01:02, 178.93it/s]


Epoch 1, Loss: 1.116629719734192


11222it [01:03, 177.91it/s]


Epoch 2, Loss: 0.9283915758132935


11222it [01:02, 178.59it/s]


Epoch 3, Loss: 0.9934739470481873


1350it [00:07, 171.18it/s]


KeyboardInterrupt: 

In [ ]:
for feature in ["STFT", "Wavelets"]:
    for model_name in ["CNN", "RNN"]:
        train_model(feature, model_name)